In [ ]:
library(STged)
library(DOTr)
library(ggplot2)
library(SeuratObject)
library(Seurat)
library(presto)
library(dplyr)
library(SeuratDisk)
library(Matrix)

In [ ]:
python_env <- "/net/data.isilon/ag-saez/yliu/SOFTWARE/.miniconda3/envs/multi/bin/python"
reticulate::use_python(python_env, required = TRUE)
reticulate::py_config()
anndata <- reticulate::import("anndata")
np <- reticulate::import("numpy")
sq <- reticulate::import("squidpy")

In [ ]:
donor = c('BCLL-8-T','BCLL-9-T','BCLL-10-T','BCLL-11-T','BCLL-12-T','BCLL-13-T')
prefix = c('c28w2r_7jne4i_','qvwc8t_2vsr67_','esvq52_nluss5_','exvyh1_66caqq_','p7hv1g_tjgmyj_','gcyl7c_cec61b_')

In [ ]:
donor <-'BCLL-8-T'

In [ ]:
dot_file <- paste0('data/spatial/DOT_output/','BCLL-9-T' , '.rds')
dot <- readRDS(dot_file)
beta <- dot@weights / rowSums(dot@weights)
beta <- beta / rowSums(beta)
#beta[beta < 0.03] <- 0
#beta <- beta / rowSums(beta)
head(beta, 5)
#rowSums(beta)

In [ ]:
colSums(dot@weights)

In [ ]:
sc <-readRDS('data/spatial/processed_data/scRNA.rds')
st <-readRDS('data/spatial/processed_data/spatial.rds')
sc

In [ ]:
coor <- read.csv("data/spatial/coordinates.csv", header = TRUE, stringsAsFactors = FALSE)

In [ ]:
#samples <- rownames(sc@meta.data)[sc$donor_id == donor&!sc$annotation_level_1 %in% c("preTC", "preBC")]
#sc_counts <- as.matrix(sc[["RNA"]]@counts[,samples])

#samples <- rownames(sc@meta.data)[sc$donor_id == donor&!sc$annotation_level_1 %in% c("preTC", "preBC")]

samples <- rownames(sc@meta.data)[!sc$annotation_level_1 %in% c("preTC", "preBC")]
sc_counts <- GetAssayData(sc, assay = "RNA", slot = "counts")[ ,samples, drop = FALSE]
sc_counts <- as.matrix(sc_counts)

In [ ]:
#label <- droplevels(sc@meta.data[sc$donor_id == donor&!sc$annotation_level_1 %in% c("preTC", "preBC"), ]$annotation_level_1)
#label<- droplevels(sc@meta.data[samples, "annotation_level_1"])
label <- droplevels(sc$annotation_level_1[samples])

#label <- droplevels(sc_hvg@meta.data[!sc_hvg$annotation_level_1 %in% c("preTC", "preBC"), ]$annotation_level_1)
print(length(label))
#label <- sc@meta.data$cell_type
table(label)

In [ ]:
samples <- rownames(st@meta.data)[st$donor_id == donor]
st_counts <- GetAssayData(st, assay = "Spatial", slot = 'counts')
st_counts <- as.matrix(st_counts[, samples, drop = FALSE])

In [ ]:
slice <- 'c28w2r_7jne4i'
xy <- coor[startsWith(coor$X, slice), ]
xy <- xy[, c("row", "col")]
xy <- as.matrix(xy)
rownames(xy) <- NULL
colnames(xy) <- NULL

In [ ]:
print(dim(sc_counts))
print(length(label))
model.est = STged(sc_exp = sc_counts, sc_label = label, 
                  spot_exp = st_counts, spot_loc = xy,  beta = beta,
                  python_env = python_env,depthscale = 1e6,
                  knei = 6,  methodL =  "Square",coord_type = "grid", 
                  lambda1 = NULL, lambda2 = NULL, cutoff = 0.05, 
                  maxiter = 500,  epsilon = 1e-5,verbose = TRUE)

In [ ]:
genes <- rownames(model.est$F_list[[1]])
out_file <- paste0('data/spatial/genes/','BCLL-9-T' , '.csv')
write.csv(
  data.frame(gene = genes),
  file = out_file,
  row.names = FALSE,
  quote = FALSE
)

In [ ]:
st_file <- paste0('data/spatial/ST_output/', 'BCLL-9-T', '.rds')
saveRDS(model.est,file = st_file)
#saveRDS(model.est, file = 'data/spatial/ST_output/donor_8.rds')